# ¡Bienvenido a los pipelines!

La librería transformers de HuggingFace ofrece APIs en dos niveles distintos.

La API de alto nivel para usar modelos de código abierto en tareas típicas de inferencia se llama "pipelines". Es increíblemente fácil de usar.

Creas un pipeline con algo como:

`mi_pipeline = pipeline("la_tarea_que_quiero_hacer")`

Y después

`resultado = mi_pipeline(mi_entrada)`

¡Y eso es todo!

Al final de este colab verás una lista de todos los pipelines.

## Antes de empezar: 2 consejos importantes para usar Colab:

**Consejo 1:**

El código de ciencia de datos suele mostrar advertencias y mensajes. En su mayoría se pueden ignorar con seguridad. Échales un vistazo, y si algo sale mal más adelante, quizá te den una pista.

**Consejo 2:**

En medio de la ejecución de un Colab, puede aparecer un error como este:

> Runtime error: CUDA is required but not available for bitsandbytes. Please consider installing [...]

¡Es un mensaje de error súper engañoso! Por favor, no intentes cambiar las versiones de los paquetes...

Esto pasa porque Google cambió tu runtime de Colab, quizá porque Google Colab estaba muy ocupado. La solución es:

1. Menú Kernel >> Disconnect and delete runtime
2. Recarga el colab desde cero y ve al menú Edit >> Clear All Outputs
3. Conéctate a una nueva T4 con el botón de arriba a la derecha
4. Selecciona "View resources" en el menú de arriba a la derecha para confirmar que tienes GPU
5. Vuelve a ejecutar las celdas del colab, de arriba hacia abajo, empezando por las instalaciones de pip

Y todo debería funcionar muy bien. Si no, ¡pregúntame!

## Una nota aparte:

Quizá ya lo sepas, pero por si acaso no estás familiarizado con la palabra "inferencia" que uso aquí:

Cuando trabajas con modelos de ciencia de datos, puedes estar haciendo 2 actividades muy distintas: **entrenamiento** e **inferencia**.

### 1. Entrenamiento

El **entrenamiento** es cuando le das a un modelo datos para que se adapte y sea mejor en una tarea en el futuro. Lo hace actualizando sus configuraciones internas: los parámetros o pesos del modelo. Si entrenas un modelo que ya había sido entrenado antes, la actividad se llama "fine-tuning" (ajuste fino).

### 2. Inferencia

La **inferencia** es cuando trabajas con un modelo que _ya fue entrenado_. Usas ese modelo para producir nuevas salidas a partir de nuevas entradas, aprovechando todo lo que aprendió durante el entrenamiento. La inferencia también se llama a veces "ejecución" o "correr un modelo".

Todo lo que hemos usado en las últimas semanas con las APIs de GPT, Claude y Gemini son ejemplos de **inferencia**. La "P" de GPT significa "Pre-trained" (preentrenado), es decir, que ya fue entrenado con datos (¡muchísimos!). En la semana 6 intentaremos hacer fine-tuning de GPT nosotros mismos.

La API de pipelines de HuggingFace sirve solo para **inferencia**: ejecutar un modelo que ya fue entrenado. En la semana 7 entrenaremos nuestro propio modelo, y necesitaremos las APIs más avanzadas de HuggingFace que veremos en la próxima clase.

Grabé esta lista de reproducción en YouTube con más información sobre parámetros, entrenamiento e inferencia:
https://www.youtube.com/playlist?list=PLWHe-9GP9SMMdl6SLaovUQF2abiLGbMjs

In [ ]:
# Las instalaciones de pip deben ir en la primera línea.
# Si el kernel se reinicia, tienes que volver a ejecutar esto.
# Gracias al estudiante Daniel M. por el arreglo extra.

!pip install -q --upgrade datasets==3.6.0 transformers==4.57.6 huggingface_hub diffusers

In [ ]:
# Revisemos la GPU: debería ser una Tesla T4

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('No conectado a una GPU')
else:
  print(gpu_info)
  if gpu_info.find('Tesla T4') >= 0:
    print("Éxito - Conectado a una T4")
  else:
    print("NO CONECTADO A UNA T4")

In [ ]:
# Imports

import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import pipeline
from diffusers import DiffusionPipeline
from datasets import load_dataset
import soundfile as sf
from IPython.display import Audio

# Nota importante: cuenta de Hugging Face

En el Día 1 creamos una cuenta GRATUITA en https://huggingface.co

### Si te lo saltaste:

¡Por favor, vuelve atrás y hazlo! Luego ve al menú de tu avatar, Tokens, y crea un token de API. ¡Y asegúrate de que tenga permisos de WRITE! Después agrégalo a los secretos de la izquierda pulsando el botón de la llave.

### Si ya lo hiciste (¡gracias!)

Pulsa el botón de la llave y activa el interruptor para que este notebook tenga acceso a tu clave de Hugging Face.

In [ ]:
hf_token = userdata.get('HF_TOKEN')
if hf_token and hf_token.startswith("hf_"):
  print("La clave de HF se ve bien por ahora")
else:
  print("La clave de HF no está configurada: pulsa la llave en la barra lateral izquierda")
login(hf_token, add_to_git_credential=True)

## Usar pipelines de Hugging Face

Una forma sencilla de hacer inferencia para tareas comunes, sin preocuparte por toda la plomería, eligiendo valores por defecto razonables.


### Cómo funciona:

PASO 1: Crea un pipeline, una función que luego puedes llamar

```python
mi_pipeline = pipeline(tarea, model=xx, device=xx)
```

Si no especificas un modelo, Hugging Face elige el modelo por defecto para esa tarea. Especifica "cuda" como device para usar una GPU NVIDIA como la de la T4. Especifica "mps" en una Mac.


PASO 2: Luego llámalo tantas veces como quieras:

```python
mi_pipeline(entrada1)
mi_pipeline(entrada2)
```

In [ ]:
# Análisis de sentimiento

mi_analizador_de_sentimiento = pipeline("sentiment-analysis", device="cuda")
resultado = mi_analizador_de_sentimiento("I'm super excited to be on the way to LLM mastery!")
print(resultado)

In [ ]:
result = my_simple_sentiment_analyzer("I should be more excited to be on the way to LLM mastery!")
print(result)

In [ ]:

better_sentiment = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment", device="cuda")
result = better_sentiment("I should be more excited to be on the way to LLM mastery!!")
print(result)

In [ ]:
# Reconocimiento de entidades nombradas (NER)

ner = pipeline("ner", device="cuda")
resultado = ner("AI Engineers are learning about the amazing pipelines from HuggingFace in Google Colab from Ed Donner")
for entidad in resultado:
  print(entidad)

In [ ]:
# Preguntas y respuestas con contexto

question="What are Hugging Face pipelines?"
context="Pipelines are a high level API for inference of LLMs with common tasks"

question_answerer = pipeline("question-answering", device="cuda")
resultado = question_answerer(question=question, context=context)
print(resultado)

In [ ]:
# Text Summarization

summarizer = pipeline("summarization", device="cuda")
text = """
The Hugging Face transformers library is an incredibly versatile and powerful tool for natural language processing (NLP).
It allows users to perform a wide range of tasks such as text classification, named entity recognition, and question answering, among others.
It's an extremely popular library that's widely used by the open-source data science community.
It lowers the barrier to entry into the field by providing Data Scientists with a productive, convenient way to work with transformer models.
"""

summary = summarizer(text, max_length=50, min_length=25, do_sample=False)
print(summary[0]['summary_text'])

In [ ]:
# Translation

translator = pipeline("translation_en_to_fr", device="cuda")
result = translator("The Data Scientists were truly amazed by the power and simplicity of the HuggingFace pipeline API.")
print(result[0]['translation_text'])

In [ ]:
# Otra traducción, mostrando cómo se especifica un modelo
# Todos los modelos de traducción están aquí: https://huggingface.co/models?pipeline_tag=translation&sort=trending

translator = pipeline("translation_en_to_es", model="Helsinki-NLP/opus-mt-en-es", device="cuda")
resultado = translator("The Data Scientists were truly amazed by the power and simplicity of the HuggingFace pipeline API.")
print(resultado[0]['translation_text'])

In [ ]:
# Clasificación

classifier = pipeline("zero-shot-classification", device="cuda")
resultado = classifier("Hugging Face's Transformers library is amazing!", candidate_labels=["technology", "sports", "politics"])
print(resultado)

In [ ]:
# Generación de texto

generator = pipeline("text-generation", device="cuda")
resultado = generator("If there's one thing I want you to remember about using HuggingFace pipelines, it's")
print(resultado[0]['generated_text'])

In [ ]:
# Generación de imágenes: ¿recuerdas esto? Ahora sabes lo que pasa por debajo
# Los pipelines también sirven para modelos de difusión, además de los de transformers

from IPython.display import display
from diffusers import AutoPipelineForText2Image
import torch

pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16")
pipe.to("cuda")
prompt = "A class of students learning AI engineering in a vibrant pop-art style"
image = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0).images[0]
display(image)

In [ ]:
# Audio Generation

from transformers import pipeline
from datasets import load_dataset
import soundfile as sf
import torch
from IPython.display import Audio

synthesiser = pipeline("text-to-speech", "microsoft/speecht5_tts", device='cuda')
embeddings_dataset = load_dataset("matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)
speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)
speech = synthesiser("Hi to an artificial intelligence engineer, on the way to mastery!", forward_params={"speaker_embeddings": speaker_embedding})

Audio(speech["audio"], rate=speech["sampling_rate"])

# Todos los pipelines disponibles

Aquí están todos los pipelines disponibles de Transformers y Diffusers.

¡Gracias al estudiante Lucky P. por sugerirme incluir esto!

Hay una lista de pipelines en las Tareas de esta página (tienes que bajar un poco y luego expandir los parámetros para ver las Tareas):

https://huggingface.co/docs/transformers/main_classes/pipelines

También está esta lista de Tareas para modelos de difusión en lugar de Transformers, siguiendo el ejemplo de generación de imágenes donde usé DiffusionPipeline arriba.

https://huggingface.co/docs/diffusers/en/api/pipelines/overview

Si encuentras ejemplos interesantes de otros pipelines, ¡compártelos conmigo! Es maravilloso cómo HuggingFace pone esta funcionalidad avanzada de IA disponible para inferencia con una API tan sencilla.